In [1]:
# Cell 1: params
from pathlib import Path
import json, csv, re
import pandas as pd
import sys

# Paths 
REPO_ROOT = Path.home() / "SIR-25-26" / "referee"
DATA_DIR = REPO_ROOT / "data"                  # target outputs
DATASET_ROOT = Path.home() / "SIR-25-26" / "FakeAVCeleb_v1.2"
META_CSV = Path.home() / "SIR-25-26" / "FakeAVCeleb_v1.2" / "meta_data.csv"
# existing original CSV/JSON 
OLD_ALL_REAL_CSV = DATA_DIR / "all_real_with_split_fixed.csv"
BACKUP_DIR = DATA_DIR / "backups_for_repair"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("META_CSV:", META_CSV)
print("DATA_DIR:", DATA_DIR)


REPO_ROOT: /home/jupyter-ashah2/SIR-25-26/referee
DATASET_ROOT: /home/jupyter-ashah2/SIR-25-26/FakeAVCeleb_v1.2
META_CSV: /home/jupyter-ashah2/SIR-25-26/FakeAVCeleb_v1.2/meta_data.csv
DATA_DIR: /home/jupyter-ashah2/SIR-25-26/referee/data


In [2]:
# Improved resolution: prefer basename matches that contain the source id folder
from collections import defaultdict
import re
from pathlib import Path
import pandas as pd
import time

DATASET_ROOT = Path.home() / "SIR-25-26" / "FakeAVCeleb_v1.2"
META_CSV = DATASET_ROOT / "meta_data.csv"

# (rebuild basename index quickly if not present)
from collections import defaultdict
basename_index = defaultdict(list)
id_folder_index = defaultdict(list)
for p in DATASET_ROOT.rglob("*.mp4"):
    basename_index[p.name].append(str(p))
    m = re.search(r"(id\d+)", str(p))
    if m:
        id_folder_index[m.group(1)].append(str(p))

meta_df = pd.read_csv(META_CSV, dtype=str, keep_default_na=False)
# detect columns
file_col = None
for c in meta_df.columns:
    if meta_df[c].str.match(r'^\d{1,6}\.mp4$', na=False).any():
        file_col = c; break
id_col = None
for c in meta_df.columns:
    if c.lower().strip() in ('source','id','source_id'):
        id_col = c; break
if id_col is None: id_col = meta_df.columns[0]

print("Detected file_col:", file_col, "id_col:", id_col)
start = time.time()
id_to_all_files = defaultdict(list)
id_to_real_files = defaultdict(list)
not_found = 0
collision_count = 0

for i,row in meta_df.iterrows():
    sid = row.get(id_col,"").strip()
    filename = None
    if file_col:
        v = row.get(file_col,"")
        if isinstance(v,str) and v.endswith(".mp4"):
            filename = v.strip()
    if not filename:
        for c in meta_df.columns:
            v = row.get(c,"")
            if isinstance(v,str) and v.endswith(".mp4"):
                filename = v.strip(); break
    resolved = None
    if filename and filename in basename_index:
        matches = basename_index[filename]
        # prefer a match that contains the sid folder
        if sid:
            preferred = [m for m in matches if f"/{sid}/" in m or f"\\{sid}\\" in m]
            if preferred:
                resolved = preferred[0]
            else:
                # if none contain sid, pick the first but count as collision
                resolved = matches[0]
                if len(matches) > 1:
                    collision_count += 1
        else:
            resolved = matches[0]
    elif sid and sid in id_folder_index and id_folder_index[sid]:
        # fallback: pick first file in that id folder
        resolved = id_folder_index[sid][0]
    else:
        # no resolved file
        not_found += 1

    if resolved:
        # classify as real if any 'RealVideo' in row
        typ = None
        for c in meta_df.columns:
            if 'type' in c.lower():
                typ = row.get(c,""); break
        id_to_all_files[sid].append(resolved)
        if typ and 'RealVideo' in typ:
            id_to_real_files[sid].append(resolved)

# deduplicate lists and remove empty keys
for k in list(id_to_all_files.keys()):
    id_to_all_files[k] = sorted(list(dict.fromkeys(id_to_all_files[k])))
    if not id_to_all_files[k]:
        del id_to_all_files[k]
for k in list(id_to_real_files.keys()):
    id_to_real_files[k] = sorted(list(dict.fromkeys(id_to_real_files[k])))
    if not id_to_real_files[k]:
        del id_to_real_files[k]

print(f"Resolved unique ids (any file): {len(id_to_all_files)}")
print(f"Resolved unique ids (real files): {len(id_to_real_files)}")
print("Rows not resolved:", not_found)
print("Filename collisions encountered (same basename across ids):", collision_count)
print("Elapsed:", time.time()-start, "s")

# Show examples of problematic collisions (if any)
if collision_count>0:
    print("\nExamples of collisions for a few basenames:")
    shown = 0
    for bnm, paths in list(basename_index.items()):
        if len(paths) > 1:
            print(bnm, "->", paths[:6])
            shown += 1
            if shown >= 6: break

# Show a few id -> real paths samples
print("\nSample id -> real paths (first 10):")
for k in list(id_to_real_files.keys())[:10]:
    print(k, id_to_real_files[k][:3])

Detected file_col: target1 id_col: source
Resolved unique ids (any file): 500
Resolved unique ids (real files): 500
Rows not resolved: 0
Filename collisions encountered (same basename across ids): 0
Elapsed: 1.547598123550415 s

Sample id -> real paths (first 10):
id00076 ['/home/jupyter-ashah2/SIR-25-26/FakeAVCeleb_v1.2/RealVideo-RealAudio/African/men/id00076/00109.mp4']
id00166 ['/home/jupyter-ashah2/SIR-25-26/FakeAVCeleb_v1.2/RealVideo-RealAudio/African/men/id00166/00010.mp4']
id00173 ['/home/jupyter-ashah2/SIR-25-26/FakeAVCeleb_v1.2/RealVideo-RealAudio/African/men/id00173/00118.mp4']
id00366 ['/home/jupyter-ashah2/SIR-25-26/FakeAVCeleb_v1.2/RealVideo-RealAudio/African/men/id00366/00118.mp4']
id00391 ['/home/jupyter-ashah2/SIR-25-26/FakeAVCeleb_v1.2/RealVideo-RealAudio/African/men/id00391/00052.mp4']
id00475 ['/home/jupyter-ashah2/SIR-25-26/FakeAVCeleb_v1.2/RealVideo-RealAudio/African/men/id00475/00099.mp4']
id00476 ['/home/jupyter-ashah2/SIR-25-26/FakeAVCeleb_v1.2/RealVideo-RealAud

In [3]:
# Cell 3: determine split for each source_id (prefer an existing CSV if present)
split_map = {}
orig_all_csv = DATA_DIR / "all_real_with_split.csv"
orig_fixed_csv = DATA_DIR / "all_real_with_split_fixed.csv"

# prefer orig_fixed if it exists (older work)
candidate_csv = None
if orig_fixed_csv.exists():
    candidate_csv = orig_fixed_csv
elif orig_all_csv.exists():
    candidate_csv = orig_all_csv

if candidate_csv:
    print("Reading existing split CSV:", candidate_csv)
    df = pd.read_csv(candidate_csv, dtype=str, keep_default_na=False)
    # prefer 'source_id' column
    sid_col = None
    for c in df.columns:
        if c.lower() in ('source_id','source','id'):
            sid_col = c
            break
    split_col = None
    for c in df.columns:
        if c.lower() == 'split':
            split_col = c
            break
    if sid_col and split_col:
        for _, r in df.iterrows():
            split_map[r[sid_col]] = r[split_col]
    print("Loaded splits for", len(split_map), "ids")
else:
    # fallback: create splits using the existing train/val jsons in data/ if present
    print("No old CSV found; using existing train/val/test JSONs to infer splits if available.")
    for name, split in [("data/train_set.json","train"),("data/val_set.json","val"),("data/test_pairs.json","test")]:
        p = REPO_ROOT / name
        if p.exists():
            try:
                with open(p, "r") as f:
                    items = json.load(f)
                    if isinstance(items, dict) and "data" in items:
                        items = items["data"]
                    for it in items:
                        sid = it.get("id") or it.get("source_id") or None
                        if sid:
                            split_map[sid] = split
            except Exception:
                pass
    print("Inferred splits for", len(split_map), "ids from JSONs")


Reading existing split CSV: /home/jupyter-ashah2/SIR-25-26/referee/data/all_real_with_split_fixed.csv
Loaded splits for 500 ids


In [4]:
# Cell 4: produce a new all_real_with_split_fixed.csv (back up old first)
out_csv = DATA_DIR / "all_real_with_split_fixed.csv"
if OLD_ALL_REAL_CSV.exists():
    OLD_ALL_REAL_CSV.rename(BACKUP_DIR / f"all_real_with_split_fixed.csv.bak")
    print("Backed up old:", OLD_ALL_REAL_CSV, "->", BACKUP_DIR)

rows = []
for src, real_paths in id_to_real_files.items():
    if not real_paths:
        continue
    split = split_map.get(src, "train")
    # we will add one row per real_path
    for p in real_paths:
        rows.append({
            "source_id": src,
            "race": "", "gender": "",   # optional; can be filled if meta_data has it
            "original_path": "",       # left blank
            "new_path": str(p),
            "split": split
        })

out_df = pd.DataFrame(rows)
out_df.to_csv(out_csv, index=False)
print("Wrote new all_real_with_split_fixed.csv rows:", len(out_df))


Backed up old: /home/jupyter-ashah2/SIR-25-26/referee/data/all_real_with_split_fixed.csv -> /home/jupyter-ashah2/SIR-25-26/referee/data/backups_for_repair
Wrote new all_real_with_split_fixed.csv rows: 500


In [5]:
# Cell 5: rebuild train_targets.json (ensures each target file exists and id matches)
pair_json = DATA_DIR / "train_set.json"
fixed_pair_json = DATA_DIR / "train_set_fixed.json"
if fixed_pair_json.exists():
    pair_json_path = fixed_pair_json
elif pair_json.exists():
    pair_json_path = pair_json
else:
    raise FileNotFoundError("No train_set.json found in data/")

with open(pair_json_path, "r") as f:
    data = json.load(f)
    items = data["data"] if isinstance(data, dict) and "data" in data else data

targets = []
missing_targets = 0
for it in items:
    tgt = it.get("file_path") or it.get("target_file")
    if not tgt:
        continue
    # If path still points to old prefix, try to locate the actual file by filename
    from pathlib import Path
    if not Path(tgt).exists():
        filename = Path(tgt).name
        matches = list(DATASET_ROOT.rglob(filename))
        if matches:
            tgt = str(matches[0])
        else:
            missing_targets += 1
            continue
    # extract id
    sid = it.get("id")
    if not sid:
        # try to infer from path
        m = re.search(r"(id\d+)", tgt)
        sid = m.group(1) if m else "unknown"
    split = split_map.get(sid, it.get("split","train"))
    targets.append({
        "file_path": tgt,
        "id": sid,
        "split": split,
        "fake_label": int(it.get("fake_label", it.get("fake", 1)))
    })

out_targets = DATA_DIR / "train_targets.json"
with open(out_targets, "w") as f:
    json.dump(targets, f, indent=2)
print("Wrote", len(targets), "targets to", out_targets, "skipped (unfound):", missing_targets)


Wrote 14003 targets to /home/jupyter-ashah2/SIR-25-26/referee/data/train_targets.json skipped (unfound): 0


In [6]:
# Cell 6: rebuild test_pairs_fixed.json and val_set_fixed.json

import json, re
from pathlib import Path

def fast_fix_pair_list(json_in, json_out, key_tgt="target_file", key_ref="reference_file"):
    with open(json_in, "r") as f:
        data = json.load(f)
        items = data["data"] if isinstance(data, dict) and "data" in data else data

    out_items = []
    missing = 0

    for it in items:
        tgt = it.get(key_tgt) or it.get("file_path")
        ref = it.get(key_ref)

        # extract id once
        sid = it.get("id")
        if not sid:
            m = re.search(r"(id\d+)", str(tgt))
            sid = m.group(1) if m else None

        # ---- fix target ----
        if not tgt or not Path(tgt).exists():
            fname = Path(tgt).name if tgt else None
            if fname and fname in basename_index:
                tgt = basename_index[fname][0]
            elif sid and sid in id_to_all_files and id_to_all_files[sid]:
                tgt = id_to_all_files[sid][0]
            else:
                missing += 1
                continue

        # ---- fix reference ----
        if not ref or not Path(ref).exists():
            chosen_ref = None
            if sid and sid in id_to_real_files and id_to_real_files[sid]:
                chosen_ref = id_to_real_files[sid][0]
            else:
                fname = Path(ref).name if ref else None
                if fname and fname in basename_index:
                    chosen_ref = basename_index[fname][0]

            if not chosen_ref:
                missing += 1
                continue
            ref = chosen_ref

        new_item = dict(it)
        new_item[key_tgt] = tgt
        new_item[key_ref] = ref
        out_items.append(new_item)

    with open(json_out, "w") as f:
        json.dump(out_items, f, indent=2)

    return len(out_items), missing


# repair test pairs
src_test = DATA_DIR / "test_pairs.json"
dst_test = DATA_DIR / "test_pairs_fixed.json"
if src_test.exists():
    ok, miss = fast_fix_pair_list(src_test, dst_test)
    print("test_pairs_fixed:", ok, "written; missing/skipped:", miss)
else:
    print("no test_pairs.json found in data/ to repair")

# repair val set
src_val = DATA_DIR / "val_set.json"
dst_val = DATA_DIR / "val_set_fixed.json"
if src_val.exists():
    ok, miss = fast_fix_pair_list(src_val, dst_val, key_tgt="file_path", key_ref="reference_file")
    print("val_set_fixed:", ok, "written; missing/skipped:", miss)
else:
    print("no val_set.json found to repair")


test_pairs_fixed: 6464 written; missing/skipped: 0
val_set_fixed: 1077 written; missing/skipped: 0


In [10]:
# Cell 7: sanity-check that each entry in train_targets.json has at least one reference in all_real_with_split_fixed.csv
import pandas as pd
from pathlib import Path
targets = json.load(open(DATA_DIR / "train_targets.json"))
real_df = pd.read_csv(DATA_DIR / "all_real_with_split_fixed.csv", dtype=str, keep_default_na=False)

# group references by (source_id, split)
ref_map = real_df.groupby(['source_id','split'])['new_path'].apply(list).to_dict()

bad_targets = []
for i, t in enumerate(targets):
    key = (t['id'], t['split'])
    if key not in ref_map or len(ref_map[key])==0:
        bad_targets.append((i, t['file_path'], key))

print("targets:", len(targets), "bad targets missing references:", len(bad_targets))
if bad_targets:
    print("first 20 bad targets:")
    for b in bad_targets[:20]:
        print(b)


targets: 14003 bad targets missing references: 0


In [11]:
# Cell 8: produce a clean train targets file that only includes targets that have at least one reference
clean = []
for t in targets:
    key = (t['id'], t['split'])
    if key in ref_map and len(ref_map[key])>0:
        clean.append(t)

out_clean = DATA_DIR / "train_targets_clean.json"
with open(out_clean, "w") as f:
    json.dump(clean, f, indent=2)
print("Wrote clean targets:", out_clean, "entries:", len(clean))


Wrote clean targets: /home/jupyter-ashah2/SIR-25-26/referee/data/train_targets_clean.json entries: 14003
